# Validação da proposta consumida pela aplicação

POC DEMO — contrato PROVISÓRIO / PENDENTE DE HOMOLOGAÇÃO. Execute em kernel novo. Credenciais somente por ambiente.


1. Configuração e preflight somente leitura. Esperado: API, OpenAPI, autenticação e SDK disponíveis.

In [ ]:
import os, json
import pandas as pd
from IPython.display import display, Markdown
from src.workbench.journey_config import Settings
cfg=Settings.load()
from src.workbench.contract_review import preflight,candidates,fetch_payload,reconcile,report
from src.workbench.sdk import source_version
from src.common.lineage import utcnow
execution_id=os.getenv("PROPOSAL_ID", "")
page_size=50
max_rows=10000
profile="v44_partial"
invoke_demo_inference=False
display({"utc":utcnow(),"code_version":source_version(),"profile":profile})
display(pd.DataFrame(preflight(cfg)))


2. Seleção controlada. Informe PROPOSAL_ID no ambiente ou execution_id nesta célula. Nenhuma proposta é selecionada automaticamente por data. Esperado: lista limitada por ciclo e UUID explícito.

In [ ]:
display(pd.DataFrame(candidates(cfg)))
if not execution_id:
    raise ValueError("Defina execution_id com um UUID da tabela acima e execute novamente desde esta célula. Não será realizado treinamento.")


3. Consumir o JSON HTTP integral com paginação, schema e hash. Esperado: páginas completas sem duplicação; este é o payload real da API.

In [ ]:
pages,offers=fetch_payload(cfg,execution_id,page_size,max_rows)
for page in pages:
    display(page)
display(pd.json_normalize(offers))


4. Regras funcionais. PASS técnico não homologa V4.4; dados ausentes devem resultar NOT_EVALUATED.

In [ ]:
display(pd.DataFrame(pages[0]["validation"]))
assert not any(rule["status"]=="FAIL" for rule in pages[0]["validation"])


5. Conferência independente de SQL, Registry e bytes RustFS/proxy; testes negativos sem alterar dados.

In [ ]:
evidence=reconcile(cfg,execution_id,pages,offers)
display(evidence)


6. Exportar evidências JSON/Markdown em artifacts/reports. A escrita é somente de relatórios.

In [ ]:
report_path=report(execution_id,pages,evidence)
display(report_path)
display(Markdown("[MLflow](http://127.0.0.1:5000) · [API](http://127.0.0.1:8000/docs) · [RustFS](http://127.0.0.1:9001)"))


7. Inferência adicional DEMO é opt-in. Não treina nem altera a proposta. Reutiliza referência explícita e entrada SQL da mesma geração.

In [ ]:
if invoke_demo_inference:
    from src.workbench.contract_review import invoke_demo
    display(invoke_demo(cfg,execution_id))
else:
    display({"optional_inference":"NOT_EXECUTED", "reason":"opt-in desativado"})
